# 🚗 Real-time Drivable Space Segmentation
**MAHE Mobility Hackathon 2026 · Track 01 · Problem Statement 2**

This notebook demonstrates inference of our custom U-Net model trained from scratch on BDD100K.

- **mIoU:** 0.6181
- **FPS:** 77.3
- **Architecture:** Custom U-Net (no pre-trained weights)


## 1. Clone repository & install dependencies

In [ ]:
!git clone https://github.com/shreejiag916-hash/drivable-space-segmentation.git
%cd drivable-space-segmentation
!pip install -q -r requirements.txt

## 2. Import libraries

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import time

from model import UNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 3. Model Architecture Overview

In [ ]:
model = UNet(num_classes=3)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'\nModel Architecture:')
print(model)

## 4. Load Pre-trained Weights
> Upload your `best_model.pth` to this Colab session, or mount Google Drive.

In [ ]:
# Option A: upload manually
# from google.colab import files
# uploaded = files.upload()  # upload best_model.pth

# Option B: load from Drive
# from google.colab import drive
# drive.mount('/content/drive')
# weights_path = '/content/drive/MyDrive/best_model.pth'

weights_path = 'checkpoints/best_model.pth'  # adjust path as needed

model.load_state_dict(torch.load(weights_path, map_location=device))
model = model.to(device).eval()
print('✔ Weights loaded successfully')

## 5. Inference on a Sample Image

In [ ]:
# ── Preprocessing ────────────────────────────────────────────
infer_transform = A.Compose([
    A.Resize(256, 512),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

CLASS_COLORS = {
    0: (0,   0,   0),
    1: (0,   255, 0),
    2: (0,   255, 255),
}

def predict(image_bgr):
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    tensor = infer_transform(image=rgb)['image'].unsqueeze(0).to(device)
    t0 = time.perf_counter()
    with torch.no_grad():
        logits = model(tensor)
    ms = (time.perf_counter() - t0) * 1000
    pred = torch.argmax(logits, dim=1).squeeze().cpu().numpy().astype(np.uint8)
    h, w = pred.shape
    colour = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, rgb_c in CLASS_COLORS.items():
        colour[pred == cls] = rgb_c
    orig_h, orig_w = image_bgr.shape[:2]
    colour_resized = cv2.resize(colour, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    orig_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(orig_rgb, 0.6, colour_resized, 0.4, 0)
    return orig_rgb, colour_resized, overlay, ms

# ── Run on sample image ──────────────────────────────────────
IMAGE_PATH = 'sample.jpg'  # replace with your image path

img = cv2.imread(IMAGE_PATH)
orig, mask, overlay, latency = predict(img)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, im, title in zip(axes,
    [orig, mask, overlay],
    ['Input Image', 'Segmentation Mask', f'Overlay  ({latency:.1f} ms)']):
    ax.imshow(im); ax.set_title(title, fontsize=13); ax.axis('off')

from matplotlib.patches import Patch
legend = [
    Patch(color=(0,0,0),     label='Background'),
    Patch(color=(0,1,0),     label='Main Drivable'),
    Patch(color=(0,1,1),     label='Alt Drivable'),
]
fig.legend(handles=legend, loc='lower center', ncol=3, fontsize=11)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig('sample_output.png', dpi=150)
plt.show()
print(f'Latency: {latency:.2f} ms | FPS: {1000/latency:.1f}')

## 6. FPS Benchmark

In [ ]:
dummy_img = np.random.randint(0, 255, (720, 1280, 3), dtype=np.uint8)
N = 100
# Warmup
for _ in range(10):
    predict(dummy_img)

t0 = time.perf_counter()
for _ in range(N):
    predict(dummy_img)
elapsed = time.perf_counter() - t0

fps = N / elapsed
print(f'{N} frames | {elapsed:.2f}s | FPS: {fps:.1f} | Latency: {elapsed/N*1000:.2f} ms')

## 7. Model Summary
| Metric | Value |
|--------|-------|
| Best mIoU | 0.6181 |
| FPS | 77.3 |
| Latency | 12.93 ms |
| Parameters | 7,763,107 |
| Architecture | Custom U-Net (scratch) |
| Dataset | BDD100K (External) |
| Loss | Dice + CrossEntropy |
| Optimizer | AdamW |
| Scheduler | CosineAnnealingLR |